In [1]:
import os
import numpy as np # type: ignore
import time
import torch
import torch.nn as nn # type: ignore
import torch.optim as optim # type: ignore
from torch.utils.data import DataLoader # type: ignore
from chess import pgn # type: ignore
from tqdm import tqdm # type: ignore

In [2]:
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games

files = [file for file in os.listdir("data") if file.endswith(".pgn")]
LIMIT_OF_FILES = min(len(files), 30)
games = []
i = 1
for file in tqdm(files):
    games.extend(load_pgn(f"data/{file}"))
    if i >= LIMIT_OF_FILES:
        break
    i += 1

 37%|███▋      | 29/79 [01:38<02:49,  3.39s/it]


In [3]:
print(f"GAMES PARSED: {len(games)}")

GAMES PARSED: 81313


In [4]:
from aux_func import create_input_for_nn

In [15]:
X, y, legal_masks = create_input_for_nn(games)

print(f"NUMBER OF SAMPLES: {len(y)}")

KeyboardInterrupt: 

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
legal_masks = torch.tensor(legal_masks, dtype=torch.bool)

In [ ]:
from dataset import ChessDataset
from model import ChessModel

In [ ]:
# Create Dataset and DataLoader
dataset = ChessDataset(X, y, legal_masks)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

# Model Initialization
model = ChessModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
num_epochs = 50
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, y_batch, legal_mask_batch in tqdm(dataloader):
        inputs = inputs.to(device)
        y_batch = y_batch.to(device)
        legal_mask_batch = legal_mask_batch.to(device)
        optimizer.zero_grad()

        move_logits = model(inputs, legal_mask=legal_mask_batch)

        loss = criterion(move_logits, y_batch)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(
        f'Epoch {epoch + 1}/{num_epochs}, '
        f'Loss: {running_loss / len(dataloader):.4f}, '
        f'Time: {minutes}m{seconds}s'
    )

In [ ]:
# Save the model
torch.save(model.state_dict(), "models/TORCH_100EPOCHS.pth")